# Progress Review 2 — MLflow Experiments

This notebook trains multiple Spark MLlib models and logs parameters/metrics to MLflow.

Goal:

Spark MLlib training → 5 MLflow experiment runs

The final project requires MLflow runs and model registry. For PR2, we demonstrate tracked experiment runs.

In [1]:
import os
import pyspark
from pyspark import SparkContext

print("PySpark package version:", pyspark.__version__)
print("SPARK_HOME:", os.environ.get("SPARK_HOME"))

existing_sc = SparkContext._active_spark_context

if existing_sc is not None:
    try:
        if existing_sc._jvm is None or existing_sc._jsc is None:
            raise RuntimeError("JVM not available – context is dead")
        existing_sc.stop()
        print("Stopped existing SparkContext.")
    except Exception as e:
        print(f"Existing SparkContext is already dead ({e}); clearing reference.")
    finally:
        SparkContext._active_spark_context = None
        SparkContext._gateway = None
        SparkContext._jvm = None

print("SparkContext cleared – safe to create a new session.")

PySpark package version: 4.0.0
SPARK_HOME: /opt/conda/lib/python3.13/site-packages/pyspark
SparkContext cleared – safe to create a new session.


In [2]:
import pandas as pd
import s3fs
import mlflow

from deltalake import DeltaTable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [3]:
spark = (
    SparkSession.builder
    .appName("aviation-pr2-mlflow-experiments")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "3g")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "aviation-jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", "4041")
    .config("spark.blockManager.port", "4042")
    .config("spark.ui.port", "4040")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark app ID:", spark.sparkContext.applicationId)

MLFLOW_TRACKING_URI = "http://mlflow:5000"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("aviation_disruption_pr2")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

Spark version: 4.0.0
Spark app ID: app-20260425121509-0027
MLflow tracking URI: http://mlflow:5000


In [4]:
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT_INTERNAL", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT},
)

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_REGION": AWS_REGION,
    "AWS_ALLOW_HTTP": "true",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

In [5]:
gold_delta_uri = "s3://lakehouse/gold/training_features_delta"

try:
    dt = DeltaTable(gold_delta_uri, storage_options=storage_options)
    features_pdf = dt.to_pandas()
    print("Read Gold features from Delta:", gold_delta_uri)

except Exception as exc:
    print("Delta read failed, falling back to Parquet.")
    print("Error:", repr(exc))

    gold_parquet_path = "s3://lakehouse/gold/training_features_parquet/training_features.parquet"
    with fs.open(gold_parquet_path, "rb") as f:
        features_pdf = pd.read_parquet(f)

print("Gold shape:", features_pdf.shape)
display(features_pdf.head())

if features_pdf.empty:
    raise RuntimeError("Gold training features table is empty.")

Read Gold features from Delta: s3://lakehouse/gold/training_features_delta
Gold shape: (72, 17)


,event_id,airport,timestamp_utc,temperature_c,wind_speed_kts,wind_speed_max_3h,wind_speed_avg_3h,precipitation_mm,precipitation_sum_3h,surface_pressure_pa,cape_j_kg,cape_avg_3h,hour_of_day,day_of_week,month,label,label_source
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,9.424164,3.638606,3.638606,3.638606,0.005761,0.005761,101271.765625,12.700439,12.700439,0,7,1,0.0,PR2_QUANTILE_FALLBACK
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,9.465088,3.977354,3.977354,3.807980,0.010565,0.016326,101288.007812,21.743164,17.221802,1,7,1,0.0,PR2_QUANTILE_FALLBACK
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,9.402130,4.554652,4.554652,4.056870,0.012485,0.028811,101208.484375,23.470459,19.304688,2,7,1,0.0,PR2_QUANTILE_FALLBACK
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,9.392700,4.152695,4.554652,4.228234,0.001440,0.024490,101199.937500,17.780762,20.998128,3,7,1,0.0,PR2_QUANTILE_FALLBACK
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,9.112518,4.710645,4.710645,4.472664,0.000959,0.014884,101201.640625,14.834229,18.695150,4,7,1,0.0,PR2_QUANTILE_FALLBACK


In [6]:
feature_cols = [
    "temperature_c",
    "wind_speed_kts",
    "wind_speed_max_3h",
    "wind_speed_avg_3h",
    "precipitation_mm",
    "precipitation_sum_3h",
    "surface_pressure_pa",
    "cape_j_kg",
    "cape_avg_3h",
    "hour_of_day",
    "day_of_week",
    "month",
]

training_df = spark.createDataFrame(features_pdf)

for col_name in feature_cols:
    training_df = training_df.withColumn(col_name, F.col(col_name).cast("double"))

training_df = training_df.withColumn("label", F.col("label").cast("double"))

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
)

model_input_df = assembler.transform(training_df).select("features", "label")

train_df, test_df = model_input_df.randomSplit([0.8, 0.2], seed=42)

if train_df.select("label").distinct().count() < 2:
    print("Train split has one class. Using full data for PR2 experiment demo.")
    train_df = model_input_df
    test_df = model_input_df

print("Train rows:", train_df.count())
print("Test rows:", test_df.count())

model_input_df.groupBy("label").count().show()

Train rows: 56
Test rows: 16
+-----+-----+
|label|count|
+-----+-----+
|  1.0|   19|
|  0.0|   53|
+-----+-----+



In [7]:
def evaluate_predictions(predictions):
    accuracy_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="accuracy",
    )

    f1_evaluator = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName="f1",
    )

    metrics = {
        "accuracy": float(accuracy_evaluator.evaluate(predictions)),
        "f1": float(f1_evaluator.evaluate(predictions)),
    }

    try:
        auc_evaluator = BinaryClassificationEvaluator(
            labelCol="label",
            rawPredictionCol="rawPrediction",
            metricName="areaUnderROC",
        )
        metrics["auc"] = float(auc_evaluator.evaluate(predictions))
    except Exception:
        metrics["auc"] = 0.0

    return metrics

In [8]:
experiments = [
    {
        "run_name": "rf_trees_5_depth_3",
        "model_type": "random_forest",
        "numTrees": 5,
        "maxDepth": 3,
    },
    {
        "run_name": "rf_trees_10_depth_3",
        "model_type": "random_forest",
        "numTrees": 10,
        "maxDepth": 3,
    },
    {
        "run_name": "rf_trees_20_depth_5",
        "model_type": "random_forest",
        "numTrees": 20,
        "maxDepth": 5,
    },
    {
        "run_name": "rf_trees_30_depth_5",
        "model_type": "random_forest",
        "numTrees": 30,
        "maxDepth": 5,
    },
    {
        "run_name": "logistic_regression_baseline",
        "model_type": "logistic_regression",
        "maxIter": 20,
        "regParam": 0.01,
    },
]

run_results = []

for experiment in experiments:
    with mlflow.start_run(run_name=experiment["run_name"]) as run:
        mlflow.log_param("model_type", experiment["model_type"])
        mlflow.log_param("feature_cols", ",".join(feature_cols))
        mlflow.log_param("train_rows", train_df.count())
        mlflow.log_param("test_rows", test_df.count())
        mlflow.log_param("spark_version", spark.version)

        if experiment["model_type"] == "random_forest":
            model = RandomForestClassifier(
                labelCol="label",
                featuresCol="features",
                numTrees=experiment["numTrees"],
                maxDepth=experiment["maxDepth"],
                seed=42,
            )

            mlflow.log_param("numTrees", experiment["numTrees"])
            mlflow.log_param("maxDepth", experiment["maxDepth"])

        elif experiment["model_type"] == "logistic_regression":
            model = LogisticRegression(
                labelCol="label",
                featuresCol="features",
                maxIter=experiment["maxIter"],
                regParam=experiment["regParam"],
            )

            mlflow.log_param("maxIter", experiment["maxIter"])
            mlflow.log_param("regParam", experiment["regParam"])

        fitted_model = model.fit(train_df)
        predictions = fitted_model.transform(test_df)

        metrics = evaluate_predictions(predictions)

        for metric_name, metric_value in metrics.items():
            mlflow.log_metric(metric_name, metric_value)

        run_results.append(
            {
                "run_id": run.info.run_id,
                "run_name": experiment["run_name"],
                **metrics,
            }
        )

        print("Completed:", experiment["run_name"], metrics)

results_df = pd.DataFrame(run_results)
display(results_df)

Completed: rf_trees_5_depth_3 {'accuracy': 1.0, 'f1': 1.0, 'auc': 1.0}
🏃 View run rf_trees_5_depth_3 at: http://mlflow:5000/#/experiments/1/runs/38383c768cbe4e178b2ba859101a4ab3
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Completed: rf_trees_10_depth_3 {'accuracy': 1.0, 'f1': 1.0, 'auc': 1.0}
🏃 View run rf_trees_10_depth_3 at: http://mlflow:5000/#/experiments/1/runs/422ea1c81c2b4cdeb54b0ad6795e4b9f
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Completed: rf_trees_20_depth_5 {'accuracy': 1.0, 'f1': 1.0, 'auc': 1.0}
🏃 View run rf_trees_20_depth_5 at: http://mlflow:5000/#/experiments/1/runs/6470f7aa6efa4d34840b91cd99ce8bb3
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Completed: rf_trees_30_depth_5 {'accuracy': 1.0, 'f1': 1.0, 'auc': 1.0}
🏃 View run rf_trees_30_depth_5 at: http://mlflow:5000/#/experiments/1/runs/5e953e77954946ceb1fc4375431ccda8
🧪 View experiment at: http://mlflow:5000/#/experiments/1
Completed: logistic_regression_baseline {'accuracy': 1

,run_id,run_name,accuracy,f1,auc
0,38383c768cbe4e178b2ba859101a4ab3,rf_trees_5_depth_3,1.0,1.0,1.0
1,422ea1c81c2b4cdeb54b0ad6795e4b9f,rf_trees_10_depth_3,1.0,1.0,1.0
2,6470f7aa6efa4d34840b91cd99ce8bb3,rf_trees_20_depth_5,1.0,1.0,1.0
3,5e953e77954946ceb1fc4375431ccda8,rf_trees_30_depth_5,1.0,1.0,1.0
4,a4d2f544a9a340eca04b0076e3d5927f,logistic_regression_baseline,1.0,1.0,1.0


In [9]:
best_run = results_df.sort_values(["f1", "accuracy"], ascending=False).iloc[0]

print("Best run:")
display(best_run.to_frame().T)

Best run:


,run_id,run_name,accuracy,f1,auc
0,38383c768cbe4e178b2ba859101a4ab3,rf_trees_5_depth_3,1.0,1.0,1.0


In [10]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("aviation_disruption_pr2")

print("Experiment ID:", experiment.experiment_id)
print("Experiment name:", experiment.name)
print("Artifact location:", experiment.artifact_location)

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

print("Number of runs found:", len(runs))

for r in runs[:5]:
    print(
        r.info.run_name,
        "run_id=",
        r.info.run_id,
        "f1=",
        r.data.metrics.get("f1"),
        "accuracy=",
        r.data.metrics.get("accuracy"),
    )

Experiment ID: 1
Experiment name: aviation_disruption_pr2
Artifact location: /mlflow/artifacts/1
Number of runs found: 20
logistic_regression_baseline run_id= a4d2f544a9a340eca04b0076e3d5927f f1= 1.0 accuracy= 1.0
rf_trees_30_depth_5 run_id= 5e953e77954946ceb1fc4375431ccda8 f1= 1.0 accuracy= 1.0
rf_trees_20_depth_5 run_id= 6470f7aa6efa4d34840b91cd99ce8bb3 f1= 1.0 accuracy= 1.0
rf_trees_10_depth_3 run_id= 422ea1c81c2b4cdeb54b0ad6795e4b9f f1= 1.0 accuracy= 1.0
rf_trees_5_depth_3 run_id= 38383c768cbe4e178b2ba859101a4ab3 f1= 1.0 accuracy= 1.0


In [11]:
best_run_name = best_run["run_name"]

print("Best run name:", best_run_name)

# Train a final Random Forest model using the best RF-style configuration.
# If logistic regression wins, still save Random Forest for consistent PR2 demo artifact.

final_model = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=20,
    maxDepth=5,
    seed=42,
).fit(model_input_df)

final_model_path = "file:///spark-models/pr2_best_random_forest_model"

final_model.write().overwrite().save(final_model_path)

print("Saved final PR2 Spark MLlib model to:", final_model_path)

Best run name: rf_trees_5_depth_3
Saved final PR2 Spark MLlib model to: file:///spark-models/pr2_best_random_forest_model


In [12]:
spark.stop()
print("Spark stopped cleanly.")

Spark stopped cleanly.
